In [27]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder
PROCESSED_DIR = Path("../data/processed")

In [28]:
df = pd.read_csv(PROCESSED_DIR/'matches_merged_cleaned.csv')
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,home_rank,home_points,away_rank,away_points,rank_diff,goal_diff
0,2001-07-18,Brazil,Paraguay,3.0,1.0,Copa América,Cali,Colombia,True,H,2.0,795.0,8.0,711.0,6.0,2.0
1,2001-07-18,Peru,Mexico,1.0,0.0,Copa América,Cali,Colombia,True,H,53.0,560.0,13.0,687.0,-40.0,1.0
2,2001-07-19,Bolivia,Costa Rica,0.0,4.0,Copa América,Medellín,Colombia,True,A,69.0,516.0,33.0,610.0,-36.0,-4.0
3,2001-07-19,Honduras,Uruguay,1.0,0.0,Copa América,Medellín,Colombia,True,H,48.0,578.0,30.0,620.0,-18.0,1.0
4,2001-07-19,Malaysia,Saudi Arabia,0.0,4.0,Friendly,Kuala Lumpur,Malaysia,False,A,109.0,404.0,29.0,621.0,-80.0,-4.0


In [29]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20914 entries, 0 to 20913
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         20914 non-null  str    
 1   home_team    20914 non-null  str    
 2   away_team    20914 non-null  str    
 3   home_score   20914 non-null  float64
 4   away_score   20914 non-null  float64
 5   tournament   20914 non-null  str    
 6   city         20914 non-null  str    
 7   country      20914 non-null  str    
 8   neutral      20914 non-null  bool   
 9   result       20914 non-null  str    
 10  home_rank    20914 non-null  float64
 11  home_points  20914 non-null  float64
 12  away_rank    20914 non-null  float64
 13  away_points  20914 non-null  float64
 14  rank_diff    20914 non-null  float64
 15  goal_diff    20914 non-null  float64
dtypes: bool(1), float64(8), str(7)
memory usage: 2.4 MB


In [30]:
df['date'] = pd.to_datetime(df['date'])

In [31]:
df.drop(columns=['city', 'country'], inplace=True) # as this columns are not useful in prediction
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20914 entries, 0 to 20913
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         20914 non-null  datetime64[us]
 1   home_team    20914 non-null  str           
 2   away_team    20914 non-null  str           
 3   home_score   20914 non-null  float64       
 4   away_score   20914 non-null  float64       
 5   tournament   20914 non-null  str           
 6   neutral      20914 non-null  bool          
 7   result       20914 non-null  str           
 8   home_rank    20914 non-null  float64       
 9   home_points  20914 non-null  float64       
 10  away_rank    20914 non-null  float64       
 11  away_points  20914 non-null  float64       
 12  rank_diff    20914 non-null  float64       
 13  goal_diff    20914 non-null  float64       
dtypes: bool(1), datetime64[us](1), float64(8), str(4)
memory usage: 2.1 MB


### Encoding

In [32]:
print(df['tournament'].nunique())
df['tournament'].value_counts().head(15)

111


tournament
Friendly                                7071
FIFA World Cup qualification            4885
UEFA Euro qualification                 1405
African Cup of Nations qualification    1163
UEFA Nations League                      584
FIFA World Cup                           472
African Cup of Nations                   446
AFC Asian Cup qualification              443
CONCACAF Nations League                  380
Gold Cup                                 279
CECAFA Cup                               271
COSAFA Cup                               268
UEFA Euro                                238
AFC Asian Cup                            218
AFF Championship                         213
Name: count, dtype: int64

In [33]:
def bucket_tournament(name: str) -> str:
    name = name.lower()
    if 'world cup' in name and 'qualification' not in name:
        return 'world_cup'
    if 'world cup qualification' in name:
        return 'world_cup_qualifier'
    if any(k in name for k in ["cup of nations", "euro", "copa am", "asian cup", "gold cup", "confederations"]):
        return 'continental_championship'
    if 'friendly' in name:
        return 'friendly'
    return 'other'

df['tournament_bucket'] = df['tournament'].apply(bucket_tournament)
df['tournament_bucket'].value_counts()

tournament_bucket
friendly                    7071
world_cup_qualifier         4885
continental_championship    4558
other                       3905
world_cup                    495
Name: count, dtype: int64

In [34]:
encoder = OneHotEncoder()
encoded_array = encoder.fit_transform(df[['tournament_bucket']])
encoded_array

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 20914 stored elements and shape (20914, 5)>

In [35]:
encoded_df = pd.DataFrame(encoded_array.toarray(), columns=encoder.get_feature_names_out(['tournament_bucket']))
df = pd.concat([df.drop(['tournament_bucket'], axis=1), encoded_df], axis=1)
df.head()

,date,home_team,away_team,home_score,away_score,tournament,neutral,result,home_rank,home_points,away_rank,away_points,rank_diff,goal_diff,tournament_bucket_continental_championship,tournament_bucket_friendly,tournament_bucket_other,tournament_bucket_world_cup,tournament_bucket_world_cup_qualifier
0,2001-07-18,Brazil,Paraguay,3.0,1.0,Copa América,True,H,2.0,795.0,8.0,711.0,6.0,2.0,1.0,0.0,0.0,0.0,0.0
1,2001-07-18,Peru,Mexico,1.0,0.0,Copa América,True,H,53.0,560.0,13.0,687.0,-40.0,1.0,1.0,0.0,0.0,0.0,0.0
2,2001-07-19,Bolivia,Costa Rica,0.0,4.0,Copa América,True,A,69.0,516.0,33.0,610.0,-36.0,-4.0,1.0,0.0,0.0,0.0,0.0
3,2001-07-19,Honduras,Uruguay,1.0,0.0,Copa América,True,H,48.0,578.0,30.0,620.0,-18.0,1.0,1.0,0.0,0.0,0.0,0.0
4,2001-07-19,Malaysia,Saudi Arabia,0.0,4.0,Friendly,False,A,109.0,404.0,29.0,621.0,-80.0,-4.0,0.0,1.0,0.0,0.0,0.0


In [36]:
result_mapping = {'A':0, 'D':1, 'H':2}
df['result_encoded'] = df['result'].map(result_mapping)
df.head()

,date,home_team,away_team,home_score,away_score,tournament,neutral,result,home_rank,home_points,away_rank,away_points,rank_diff,goal_diff,tournament_bucket_continental_championship,tournament_bucket_friendly,tournament_bucket_other,tournament_bucket_world_cup,tournament_bucket_world_cup_qualifier,result_encoded
0,2001-07-18,Brazil,Paraguay,3.0,1.0,Copa América,True,H,2.0,795.0,8.0,711.0,6.0,2.0,1.0,0.0,0.0,0.0,0.0,2
1,2001-07-18,Peru,Mexico,1.0,0.0,Copa América,True,H,53.0,560.0,13.0,687.0,-40.0,1.0,1.0,0.0,0.0,0.0,0.0,2
2,2001-07-19,Bolivia,Costa Rica,0.0,4.0,Copa América,True,A,69.0,516.0,33.0,610.0,-36.0,-4.0,1.0,0.0,0.0,0.0,0.0,0
3,2001-07-19,Honduras,Uruguay,1.0,0.0,Copa América,True,H,48.0,578.0,30.0,620.0,-18.0,1.0,1.0,0.0,0.0,0.0,0.0,2
4,2001-07-19,Malaysia,Saudi Arabia,0.0,4.0,Friendly,False,A,109.0,404.0,29.0,621.0,-80.0,-4.0,0.0,1.0,0.0,0.0,0.0,0


### Building New Features

In [37]:
df.columns

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'neutral', 'result', 'home_rank', 'home_points',
       'away_rank', 'away_points', 'rank_diff', 'goal_diff',
       'tournament_bucket_continental_championship',
       'tournament_bucket_friendly', 'tournament_bucket_other',
       'tournament_bucket_world_cup', 'tournament_bucket_world_cup_qualifier',
       'result_encoded'],
      dtype='str')

In [38]:
df['home_score'] = df['home_score'].astype(int)
df['away_score'] = df['away_score'].astype(int)

In [39]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,neutral,result,home_rank,home_points,away_rank,away_points,rank_diff,goal_diff,tournament_bucket_continental_championship,tournament_bucket_friendly,tournament_bucket_other,tournament_bucket_world_cup,tournament_bucket_world_cup_qualifier,result_encoded
0,2001-07-18,Brazil,Paraguay,3,1,Copa América,True,H,2.0,795.0,8.0,711.0,6.0,2.0,1.0,0.0,0.0,0.0,0.0,2
1,2001-07-18,Peru,Mexico,1,0,Copa América,True,H,53.0,560.0,13.0,687.0,-40.0,1.0,1.0,0.0,0.0,0.0,0.0,2
2,2001-07-19,Bolivia,Costa Rica,0,4,Copa América,True,A,69.0,516.0,33.0,610.0,-36.0,-4.0,1.0,0.0,0.0,0.0,0.0,0
3,2001-07-19,Honduras,Uruguay,1,0,Copa América,True,H,48.0,578.0,30.0,620.0,-18.0,1.0,1.0,0.0,0.0,0.0,0.0,2
4,2001-07-19,Malaysia,Saudi Arabia,0,4,Friendly,False,A,109.0,404.0,29.0,621.0,-80.0,-4.0,0.0,1.0,0.0,0.0,0.0,0


In [40]:
home_df = df[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'result']].copy()
home_df.columns = ['date', 'team', 'opponent', 'goal_for', 'goal_against', 'result']
home_df['won'] = (home_df['result'] == 'H').astype(int)
home_df.head()

,date,team,opponent,goal_for,goal_against,result,won
0,2001-07-18,Brazil,Paraguay,3,1,H,1
1,2001-07-18,Peru,Mexico,1,0,H,1
2,2001-07-19,Bolivia,Costa Rica,0,4,A,0
3,2001-07-19,Honduras,Uruguay,1,0,H,1
4,2001-07-19,Malaysia,Saudi Arabia,0,4,A,0


In [41]:
away_df = df[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'result']].copy()
away_df.columns = ['date', 'opponent', 'team', 'goal_against', 'goal_for', 'result']
away_df['won'] = (df['result'] == 'A').astype(int)
away_df.head()

,date,opponent,team,goal_against,goal_for,result,won
0,2001-07-18,Brazil,Paraguay,3,1,H,0
1,2001-07-18,Peru,Mexico,1,0,H,0
2,2001-07-19,Bolivia,Costa Rica,0,4,A,1
3,2001-07-19,Honduras,Uruguay,1,0,H,0
4,2001-07-19,Malaysia,Saudi Arabia,0,4,A,1


In [42]:
full_df = pd.concat([home_df, away_df], ignore_index=True)
full_df.head()

,date,team,opponent,goal_for,goal_against,result,won
0,2001-07-18,Brazil,Paraguay,3,1,H,1
1,2001-07-18,Peru,Mexico,1,0,H,1
2,2001-07-19,Bolivia,Costa Rica,0,4,A,0
3,2001-07-19,Honduras,Uruguay,1,0,H,1
4,2001-07-19,Malaysia,Saudi Arabia,0,4,A,0


In [43]:
print(home_df.shape)
print(away_df.shape)
print(full_df.shape)

(20914, 7)
(20914, 7)
(41828, 7)


#### Feature 1: form_win_rate

In [44]:
full_df['form_win_rate'] = full_df['won'].shift(1).rolling(window=5, min_periods=1).mean().round(2)
full_df.head()

,date,team,opponent,goal_for,goal_against,result,won,form_win_rate
0,2001-07-18,Brazil,Paraguay,3,1,H,1,NaN
1,2001-07-18,Peru,Mexico,1,0,H,1,1.00
2,2001-07-19,Bolivia,Costa Rica,0,4,A,0,1.00
3,2001-07-19,Honduras,Uruguay,1,0,H,1,0.67
4,2001-07-19,Malaysia,Saudi Arabia,0,4,A,0,0.75


#### Feature 2: form_goal_for

In [45]:
full_df['form_goal_for'] = full_df['goal_for'].shift(1).rolling(window=5, min_periods=1).mean().round(2)
full_df.head()

,date,team,opponent,goal_for,goal_against,result,won,form_win_rate,form_goal_for
0,2001-07-18,Brazil,Paraguay,3,1,H,1,NaN,NaN
1,2001-07-18,Peru,Mexico,1,0,H,1,1.00,3.00
2,2001-07-19,Bolivia,Costa Rica,0,4,A,0,1.00,2.00
3,2001-07-19,Honduras,Uruguay,1,0,H,1,0.67,1.33
4,2001-07-19,Malaysia,Saudi Arabia,0,4,A,0,0.75,1.25


#### Feature 3: form_goal_against

In [46]:
full_df['form_goal_against'] = full_df['goal_against'].shift(1).rolling(window=5, min_periods=1).mean().round()
full_df.head()

,date,team,opponent,goal_for,goal_against,result,won,form_win_rate,form_goal_for,form_goal_against
0,2001-07-18,Brazil,Paraguay,3,1,H,1,NaN,NaN,NaN
1,2001-07-18,Peru,Mexico,1,0,H,1,1.00,3.00,1.0
2,2001-07-19,Bolivia,Costa Rica,0,4,A,0,1.00,2.00,0.0
3,2001-07-19,Honduras,Uruguay,1,0,H,1,0.67,1.33,2.0
4,2001-07-19,Malaysia,Saudi Arabia,0,4,A,0,0.75,1.25,1.0


In [47]:
form_cols = ['date', 'team', 'form_win_rate', 'form_goal_for', 'form_goal_against']
home_form = full_df[full_df['team'].isin(df['home_team'])][form_cols]
home_form.columns = ['date','home_team', 'home_form_win_rate', 'home_form_goal_for', 'home_form_goal_against']

away_form = full_df[full_df['team'].isin(df['away_team'])][form_cols]
away_form.columns = ['date','away_team', 'away_form_win_rate', 'away_form_goal_for', 'away_form_goal_against']
away_form.shape

(41797, 5)

In [48]:
df = df.merge(home_form, on=['date', 'home_team'], how='left')
df = df.merge(away_form, on=['date', 'away_team'], how='left')
df.head()

,date,home_team,away_team,home_score,away_score,tournament,neutral,result,home_rank,home_points,...,tournament_bucket_other,tournament_bucket_world_cup,tournament_bucket_world_cup_qualifier,result_encoded,home_form_win_rate,home_form_goal_for,home_form_goal_against,away_form_win_rate,away_form_goal_for,away_form_goal_against
0,2001-07-18,Brazil,Paraguay,3,1,Copa América,True,H,2.0,795.0,...,0.0,0.0,0.0,2,NaN,NaN,NaN,0.6,1.6,1.0
1,2001-07-18,Peru,Mexico,1,0,Copa América,True,H,53.0,560.0,...,0.0,0.0,0.0,2,1.00,3.00,1.0,0.6,1.6,1.0
2,2001-07-19,Bolivia,Costa Rica,0,4,Copa América,True,A,69.0,516.0,...,0.0,0.0,0.0,0,1.00,2.00,0.0,0.4,1.0,1.0
3,2001-07-19,Honduras,Uruguay,1,0,Copa América,True,H,48.0,578.0,...,0.0,0.0,0.0,2,0.67,1.33,2.0,0.6,1.8,1.0
4,2001-07-19,Malaysia,Saudi Arabia,0,4,Friendly,False,A,109.0,404.0,...,0.0,0.0,0.0,0,0.75,1.25,1.0,0.4,1.4,1.0


#### Feature 4: h2h_win (Head To Head Win)

In [49]:
def create_h2h(df : pd.DataFrame) -> pd.DataFrame:
    h2h_rates = []
    history = {}  # key: frozenset({teamA, teamB}) -> list of (date, winner)

    for _,row in df.iterrows():
        key = frozenset([row['home_team'], row['away_team']])
        past = history.get(key, [])
        if past:
            home_wins = sum(1 for (_, winner) in past if winner==row['home_team'])
            h2h_rates.append(home_wins/len(past))
        else:
            h2h_rates.append(np.nan)
        
        if row['result'] == 'H':
            winner = row['home_team']
        elif row['result'] == 'A':
            winner = row['away_team']
        else:
            winner = None
        history.setdefault(key, []).append((row['date'], winner))
    
    df['h2h_win_rate'] = h2h_rates
    return df

df = create_h2h(df)
df.head()

,date,home_team,away_team,home_score,away_score,tournament,neutral,result,home_rank,home_points,...,tournament_bucket_world_cup,tournament_bucket_world_cup_qualifier,result_encoded,home_form_win_rate,home_form_goal_for,home_form_goal_against,away_form_win_rate,away_form_goal_for,away_form_goal_against,h2h_win_rate
0,2001-07-18,Brazil,Paraguay,3,1,Copa América,True,H,2.0,795.0,...,0.0,0.0,2,NaN,NaN,NaN,0.6,1.6,1.0,NaN
1,2001-07-18,Peru,Mexico,1,0,Copa América,True,H,53.0,560.0,...,0.0,0.0,2,1.00,3.00,1.0,0.6,1.6,1.0,NaN
2,2001-07-19,Bolivia,Costa Rica,0,4,Copa América,True,A,69.0,516.0,...,0.0,0.0,0,1.00,2.00,0.0,0.4,1.0,1.0,NaN
3,2001-07-19,Honduras,Uruguay,1,0,Copa América,True,H,48.0,578.0,...,0.0,0.0,2,0.67,1.33,2.0,0.6,1.8,1.0,NaN
4,2001-07-19,Malaysia,Saudi Arabia,0,4,Friendly,False,A,109.0,404.0,...,0.0,0.0,0,0.75,1.25,1.0,0.4,1.4,1.0,NaN


In [50]:
df.isnull().sum()

date                                             0
home_team                                        0
away_team                                        0
home_score                                       0
away_score                                       0
tournament                                       0
neutral                                          0
result                                           0
home_rank                                        0
home_points                                      0
away_rank                                        0
away_points                                      0
rank_diff                                        0
goal_diff                                        0
tournament_bucket_continental_championship       0
tournament_bucket_friendly                       0
tournament_bucket_other                          0
tournament_bucket_world_cup                      0
tournament_bucket_world_cup_qualifier            0
result_encoded                 

In [51]:
# h2h: missing means "never played before" — 0.5 (neutral / unknown) is a reasonable default
df['h2h_win_rate'] = df['h2h_win_rate'].fillna(0.5)

In [52]:
df.dropna(inplace=True)
df.isnull().sum()

date                                          0
home_team                                     0
away_team                                     0
home_score                                    0
away_score                                    0
tournament                                    0
neutral                                       0
result                                        0
home_rank                                     0
home_points                                   0
away_rank                                     0
away_points                                   0
rank_diff                                     0
goal_diff                                     0
tournament_bucket_continental_championship    0
tournament_bucket_friendly                    0
tournament_bucket_other                       0
tournament_bucket_world_cup                   0
tournament_bucket_world_cup_qualifier         0
result_encoded                                0
home_form_win_rate                      

### Final Dataset for model traning

In [53]:
df.columns

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'neutral', 'result', 'home_rank', 'home_points',
       'away_rank', 'away_points', 'rank_diff', 'goal_diff',
       'tournament_bucket_continental_championship',
       'tournament_bucket_friendly', 'tournament_bucket_other',
       'tournament_bucket_world_cup', 'tournament_bucket_world_cup_qualifier',
       'result_encoded', 'home_form_win_rate', 'home_form_goal_for',
       'home_form_goal_against', 'away_form_win_rate', 'away_form_goal_for',
       'away_form_goal_against', 'h2h_win_rate'],
      dtype='str')

In [54]:
feature_cols = ['neutral', 'home_rank', 'home_points',
       'away_rank', 'away_points', 'rank_diff',
       'tournament_bucket_continental_championship',
       'tournament_bucket_friendly', 'tournament_bucket_other',
       'tournament_bucket_world_cup', 'tournament_bucket_world_cup_qualifier', 'home_form_win_rate', 'home_form_goal_for',
       'home_form_goal_against', 'away_form_win_rate', 'away_form_goal_for',
       'away_form_goal_against', 'h2h_win_rate']

reference_cols = ['date', 'home_team', 'away_team']

traget_col = ['result_encoded']

final_df = df[feature_cols + reference_cols + traget_col].copy()
final_df.head()

,neutral,home_rank,home_points,away_rank,away_points,rank_diff,tournament_bucket_continental_championship,tournament_bucket_friendly,tournament_bucket_other,tournament_bucket_world_cup,...,home_form_goal_for,home_form_goal_against,away_form_win_rate,away_form_goal_for,away_form_goal_against,h2h_win_rate,date,home_team,away_team,result_encoded
1,True,53.0,560.0,13.0,687.0,-40.0,1.0,0.0,0.0,0.0,...,3.00,1.0,0.6,1.6,1.0,0.5,2001-07-18,Peru,Mexico,2
2,True,69.0,516.0,33.0,610.0,-36.0,1.0,0.0,0.0,0.0,...,2.00,0.0,0.4,1.0,1.0,0.5,2001-07-19,Bolivia,Costa Rica,0
3,True,48.0,578.0,30.0,620.0,-18.0,1.0,0.0,0.0,0.0,...,1.33,2.0,0.6,1.8,1.0,0.5,2001-07-19,Honduras,Uruguay,2
4,False,109.0,404.0,29.0,621.0,-80.0,0.0,1.0,0.0,0.0,...,1.25,1.0,0.4,1.4,1.0,0.5,2001-07-19,Malaysia,Saudi Arabia,0
5,False,82.0,483.0,45.0,584.0,-37.0,0.0,1.0,0.0,0.0,...,1.00,2.0,0.4,1.8,1.0,0.5,2001-07-20,Cuba,Jamaica,1


In [55]:
output_path = PROCESSED_DIR/'model_features.csv'
final_df.to_csv(output_path, index=False)
print(f'Dataset saved to {output_path}')

Dataset saved to ..\data\processed\model_features.csv
